In [1]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_cohere import CohereRerank
from dotenv import load_dotenv

load_dotenv()

C:\Users\gurut\AppData\Local\Temp\ipykernel_16020\2727222791.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


True

In [2]:
%pip install langchain_cohere

Note: you may need to restart the kernel to use updated packages.


c:\Users\gurut\Documents\projects\RAG\.venv\Scripts\python.exe: No module named pip


SETUP: Create expanded sample company data (50+ chunks)

In [3]:
chunks = [
    # Tesla - Financial & Production
    "Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.",
    "Tesla's automotive gross margin improved to 19.3% this quarter.",
    "Tesla Cybertruck production ramp begins in 2024 with initial deliveries.",
    "Tesla announced plans to expand Gigafactory production capacity.",
    "Tesla stock price reached new highs following earnings announcement.",
    "Tesla's energy storage business grew 40% year-over-year.",
    "Tesla continues to lead in electric vehicle market share globally.",
    "Tesla Model Y became the best-selling vehicle worldwide.",
    "Tesla reported strong free cash flow generation of $7.5 billion.",
    "Tesla's Full Self-Driving revenue increased significantly.",
    
    # Microsoft - Development & Acquisitions
    "Microsoft acquired GitHub for $7.5 billion in 2018.",
    "Microsoft's cloud revenue Azure grew 29% year-over-year.",
    "Microsoft announced new AI features for Visual Studio Code.",
    "Microsoft Teams integration with GitHub enhances developer workflow.",
    "Microsoft's developer tools division sees strong adoption.",
    "Microsoft acquired Activision Blizzard for $68.7 billion.",
    "Microsoft's productivity suite gained 50 million new users.",
    "Microsoft announced new Surface devices for developers.",
    "Microsoft's AI Copilot features expand to more development tools.",
    "Microsoft's enterprise solutions drive revenue growth.",
    
    # NVIDIA - AI & Hardware
    "NVIDIA's data center revenue reached $47.5 billion annually.",
    "NVIDIA's H100 GPUs see unprecedented demand for AI training.",
    "NVIDIA announced next-generation Blackwell architecture.",
    "NVIDIA's gaming revenue declined due to crypto market changes.",
    "NVIDIA's automotive AI platform partnerships expanded.",
    "NVIDIA's AI chip shortage affects cloud providers.",
    "NVIDIA stock valuation exceeds $2 trillion market cap.",
    "NVIDIA's CUDA platform dominates AI development.",
    "NVIDIA announced new AI inference chips for edge computing.",
    "NVIDIA's partnership with major cloud providers strengthens.",
    
    # Google/Alphabet - AI & Cloud
    "Google's AI investments total over $100 billion in recent years.",
    "Google Cloud revenue grew 35% reaching $8.4 billion quarterly.",
    "Google announced Gemini AI model competing with GPT-4.",
    "Google's search advertising revenue remains strong at $59 billion.",
    "Google's Workspace products integrate advanced AI features.",
    "Google announced quantum computing breakthroughs.",
    "Google's autonomous vehicle division Waymo expands operations.",
    "Google's AI research published breakthrough papers.",
    "Google's cloud AI services see enterprise adoption.",
    "Google faces regulatory scrutiny over AI dominance.",
    
    # Noisy/Less Relevant Chunks
    "The Tesla coil was invented by Nikola Tesla in 1891.",
    "Microsoft Excel spreadsheet formulas can be complex for beginners.",
    "NVIDIA Shield TV streaming device gets software update.",
    "Google Maps navigation improved with real-time traffic data.",
    "Production delays affected multiple manufacturing sectors.",
    "Financial markets showed volatility during earnings season.",
    "Revenue recognition standards changed for software companies.",
    "Hardware components face supply chain constraints globally.",
    "Development tools market grows with remote work trends.",
    "AI research requires significant computational resources.",
    "Quarterly reports show mixed results across tech sector.",
    "Stock market analysts upgrade technology sector ratings.",
    "Cloud computing adoption accelerates in enterprise market.",
    "Data center construction increases globally.",
    "Semiconductor shortage impacts various industries.",
    "Electric vehicle charging infrastructure expands rapidly.",
    "Software development productivity tools gain popularity.",
    "Machine learning frameworks become more accessible.",
    "Enterprise software licensing models evolve.",
    "Technology conferences showcase latest innovations."
]

print(f"Created {len(chunks)} sample chunks for demonstration")


Created 60 sample chunks for demonstration


In [4]:
# Convert to Document objects
documents = [Document(page_content=chunk, metadata={"source": f"chunk_{i}"}) for i, chunk in enumerate(chunks)]

 
SETUP: Create the three types of retrievers

# 1. Vector Retriever (Semantic Search/Dense Retrieval)


Setting up hybrid retriever...

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Initialize local embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

# Create Chroma vector store
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
)

# Create vector retriever
vector_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 15}
)

c:\Users\gurut\Documents\projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4173.72it/s]


# 2. BM25 Retriever (Keyword Search/Sparse Retrieval)

In [7]:
# 2. BM25 Retriever
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 15

# 3. Hybrid Retriever


In [8]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.7, 0.3]
)

print("Setup complete!\n")


Setup complete!



Step 1: Get results from hybrid search (retrieve more chunks)


In [9]:
query = "Tesla financial performance and production updates"

print("STEP 1: Hybrid Search Results")
print("-"*50)

retrieved_docs = hybrid_retriever.invoke(query)  # Get top 25 for reranking

# Show top 10 from hybrid search
for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i:2d}. {doc.page_content}")

print(f"\n(Retrieved {len(retrieved_docs)} total chunks for reranking)\n")


STEP 1: Hybrid Search Results
--------------------------------------------------
 1. Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
 2. Tesla announced plans to expand Gigafactory production capacity.
 3. Tesla reported strong free cash flow generation of $7.5 billion.
 4. Tesla stock price reached new highs following earnings announcement.
 5. Tesla continues to lead in electric vehicle market share globally.
 6. Tesla Cybertruck production ramp begins in 2024 with initial deliveries.
 7. Tesla Model Y became the best-selling vehicle worldwide.
 8. The Tesla coil was invented by Nikola Tesla in 1891.
 9. Financial markets showed volatility during earnings season.
10. Tesla's automotive gross margin improved to 19.3% this quarter.
11. Tesla's energy storage business grew 40% year-over-year.
12. Tesla's Full Self-Driving revenue increased significantly.
13. Stock market analysts upgrade technology sector ratings.
14. Production delays affected multiple manufacturin

Step 2: Apply Cohere Reranking

In [13]:
from sentence_transformers import CrossEncoder

print("STEP 2: After Local Reranking (Top 10)")
print("-" * 50)

# Initialize the local cross-encoder reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Prepare query-document pairs
pairs = [
    (query, doc.page_content)
    for doc in retrieved_docs
]

# Calculate relevance scores
scores = reranker.predict(pairs)

# Sort documents by relevance score (highest first)
ranked_results = sorted(
    zip(retrieved_docs, scores),
    key=lambda item: item[1],
    reverse=True
)

# Select the top 10 documents
reranked_docs = [
    doc for doc, score in ranked_results[:10]
]

# Display reranked results
for i, (doc, score) in enumerate(ranked_results[:10], 1):
    print(f"{i:2d}. Score: {score:.4f}")
    print(doc.page_content)
    print("-" * 50)

print("\n" + "=" * 80)
print("ANALYSIS:")
print("✅ Hybrid Search: Retrieved candidate documents")
print("✅ Local Reranking: Cross-encoder evaluated query-document relevance")
print("✅ Final Results: Documents sorted by relevance score")

# Compare before and after reranking
print("\n" + "=" * 80)
print("KEY IMPROVEMENTS AFTER RERANKING:")
print("-" * 40)

hybrid_top_5 = retrieved_docs[:5]
reranked_top_5 = reranked_docs[:5]

print("\nBEFORE (Hybrid Top 5):")
for i, doc in enumerate(hybrid_top_5, 1):
    print(f"{i}. {doc.page_content}")

print("\nAFTER (Reranked Top 5):")
for i, doc in enumerate(reranked_top_5, 1):
    print(f"{i}. {doc.page_content}")

STEP 2: After Local Reranking (Top 10)
--------------------------------------------------


c:\Users\gurut\Documents\projects\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gurut\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\gurut\Documents\projects\RAG\.venv\Lib\site-packages\huggingface_hub\file_downl

 1. Score: -2.0369
Tesla Cybertruck production ramp begins in 2024 with initial deliveries.
--------------------------------------------------
 2. Score: -2.0868
Tesla announced plans to expand Gigafactory production capacity.
--------------------------------------------------
 3. Score: -2.9066
Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
--------------------------------------------------
 4. Score: -3.2088
Tesla reported strong free cash flow generation of $7.5 billion.
--------------------------------------------------
 5. Score: -3.6847
Tesla's automotive gross margin improved to 19.3% this quarter.
--------------------------------------------------
 6. Score: -6.0221
Tesla continues to lead in electric vehicle market share globally.
--------------------------------------------------
 7. Score: -6.0331
Tesla's Full Self-Driving revenue increased significantly.
--------------------------------------------------
 8. Score: -6.3075
Tesla stock price reached new

FINAL RAG: Generate answer with reranked context

In [15]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

print("\n" + "=" * 80)
print("FINAL: RAG with Locally Reranked Context")
print("-" * 40)

# Use top 5 reranked documents for final answer
top_reranked = reranked_docs[:5]

combined_input = f"""
Based on the following documents, please answer this question:

Question: {query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in top_reranked])}

Instructions:
1. Provide a clear and helpful answer.
2. Use only the information available in the provided documents.
3. Do not introduce external information or make assumptions.
4. If the documents do not contain enough information, explicitly say so.
"""

# Initialize local Ollama model
model = ChatOllama(
    model="llama3.2",
    temperature=0
)

messages = [
    SystemMessage(
        content="""You are a helpful RAG assistant.
        Answer questions accurately using only the provided context.
        If the context is insufficient, acknowledge that you don't have enough information."""
    ),
    HumanMessage(content=combined_input),
]

# Generate final answer
result = model.invoke(messages)

print("Generated Response:")
print(result.content)


FINAL: RAG with Locally Reranked Context
----------------------------------------
Generated Response:
Based on the provided documents, here are the answers to the question about Tesla financial performance and production updates:

1. Tesla's financial performance:
   - Record quarterly revenue: $25.2 billion in Q3 2024.
   - Strong free cash flow generation: $7.5 billion.

2. Tesla's production updates:
   - The production ramp of the Tesla Cybertruck begins in 2024 with initial deliveries.
   - Tesla plans to expand Gigafactory production capacity.

Note that the documents do not provide specific details on the automotive gross margin improvement percentage (19.3%) beyond stating it occurred this quarter.
